# D2. A curve number from Earth Engine

This validation record applies the spatial curve-number workflow to Difficult Run near Great Falls, Virginia. It was executed on 6 August 2026 with `cnkit` 1.1.0, a registered Earth Engine project, USGS watershed delineation, Annual NLCD, and Soil Data Access.

The analysis estimates the observed joint distribution of land cover and hydrologic soil group, calculates a composite curve number, and retains the source, scale, coverage, and unmapped-area fields needed for interpretation. The reference basin generally completes in two to four minutes.

Notebook D1 supplies the watershed boundary used here. The executed outputs below support the values reported in the workshop materials.


In [1]:
# Portable setup: works in Colab, JupyterLab, JupyterHub and VS Code.
import importlib
import subprocess
import sys

CNKIT_EXAMPLE_VERSION = "1.1.0"

try:
    import cnkit as _cnkit_setup
    import ee as _ee_setup
except ImportError:
    _cnkit_setup = None

if getattr(_cnkit_setup, "__version__", None) != CNKIT_EXAMPLE_VERSION:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "cnkit[gee]==" + CNKIT_EXAMPLE_VERSION])
    importlib.invalidate_caches()
    for _name in list(sys.modules):
        if _name == "cnkit" or _name.startswith("cnkit."):
            sys.modules.pop(_name, None)

print("cnkit Earth Engine environment ready")

cnkit Earth Engine environment ready


In [2]:
import os
from getpass import getpass

# Runtime input keeps the project id out of notebook source and Git history.
EE_PROJECT = (os.environ.get("CNKIT_EE_PROJECT") or
              getpass("Earth Engine project id: "))

Earth Engine project id: ··········


In [3]:
import warnings
from pathlib import Path

import pandas as pd

import cnkit
from cnkit.delineate import Watershed, watershed_from_point
from cnkit.gee import Basin, initialise
from cnkit.gee._assets import CSRL_HSG_CODES
from cnkit.lookup import NLCD_CLASSES, composite_from_areas

pd.set_option("display.width", 100)
print("cnkit", cnkit.__version__)

cnkit 1.1.0


## Start a session

`initialise` is a thin wrapper over `ee.Initialize` that turns its two common
failures into named exceptions: `EarthEngineNotInstalled`, which carries the
exact install command, and `EarthEngineNotInitialised`, which chains the
original error. `cnkit` never bundles, stores or transmits a credential.

The next cell uses the same authentication flow in local Jupyter and Colab.
Expect a dict with `initialised` True and your runtime project id. If it
raises after authentication, check that the Earth Engine API is enabled on
the project and that its registration is current.

In [4]:
import ee

ee.Authenticate()
session = initialise(project=EE_PROJECT)
session

{'project': 'cee398fall2024', 'initialised': True, 'opts': {}}

## The boundary from notebook 1

`Basin` takes a `Watershed`, or any GeoJSON that `Watershed.from_geojson`
accepts. Reading the file that notebook 1 wrote means the Earth Engine work
below runs over exactly the boundary that was checked against the published
USGS drainage area, rather than over a redelineation that might differ by a
catchment.

If the file is not there, the fallback delineates it again. That call does not
touch Earth Engine: delineation is plain HTTP against USGS, because Earth
Engine ships no flow routing, no flow accumulation and no sink filling.

Expect **57.82 square miles** and 2129 vertices, well under the 5000 vertex
ceiling, so no simplification happens and no warning is raised.

In [5]:
path = Path("difficult_run.geojson")
if path.exists():
    watershed = Watershed.from_file(path)
else:
    watershed = watershed_from_point(38.97594, -77.24581)

basin = Basin(watershed, project=EE_PROJECT)
print(basin)
print("vertices", basin.watershed.vertex_count())

Basin(area_sqmi=57.822, method='nldi_splitcatchment', vertices=2129, scale=30 m)
vertices 2129


## Which years exist

Never hardcode the range. Annual NLCD sits in the Earth Engine **community**
catalog under `projects/sat-io/...`, not in the official Google catalog, and
its contents can change independently of a package release. `available_years` reads the
collection itself and caches the answer on the instance.

It held **1985 to 2025**, forty-one years, when validated on 6 August 2026.
The notebook queries it live instead of assuming that range, so a future
mirror update is reported rather than silently relabelled.

In [6]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    years = basin.available_years()

print("years %d to %d, %d of them" % (years[0], years[-1], len(years)))
for entry in caught:
    print("\n%s: %s" % (entry.category.__name__, entry.message))

years 1985 to 2025, 41 of them


## Land cover, 2001 and 2019

One `reduceRegion` per year with a frequency histogram reducer, which returns
every class area in a single call. The alternative, masking to each class in
turn, costs sixteen requests for the same answer.

Masked pixels are unmasked to 250, the documented Annual NLCD NoData value, so
a basin partly outside the CONUS footprint reports the hole instead of quietly
shrinking its own denominator. Percentages sum to 100 within each year
including NoData and including any class with no curve number.

Expect: a table of about a dozen classes per year, summing to 100 within each
year. Difficult Run is a suburban Fairfax County basin, so the developed
classes 21 to 24 should be a large share of it and deciduous forest should be
the largest single natural class. Expect the **2001 and 2019 columns to look
almost identical**. That is not a bug and it is the entire subject of notebook
3: eighteen years of suburban development moves this table very little.

In [7]:
cover = basin.landcover(years=[2001, 2019])
print(cover.groupby("year")["pct"].sum().round(6).to_string())
cover.head()

year
2001    100.0
2019    100.0


,year,nlcd,pct
0,2001,11,0.457342
1,2001,21,39.740059
2,2001,22,19.882578
3,2001,23,8.229744
4,2001,24,1.652589


In [8]:
table = cover.pivot(index="nlcd", columns="year", values="pct").fillna(0.0)
table.insert(0, "class", [NLCD_CLASSES.get(int(c), "no name in cnkit.lookup")
                          for c in table.index])
table["change"] = table[2019] - table[2001]
table.round(3).sort_values("change", key=abs, ascending=False)

year,class,2001,2019,change
nlcd,,,,
22,"Developed, Low Intensity",19.883,21.626,1.743
41,Deciduous Forest,20.519,18.835,-1.684
23,"Developed, Medium Intensity",8.230,9.376,1.147
81,Pasture/Hay,1.686,0.671,-1.015
24,"Developed, High Intensity",1.653,2.177,0.524
43,Mixed Forest,3.632,3.156,-0.477
21,"Developed, Open Space",39.740,39.615,-0.125
42,Evergreen Forest,0.119,0.064,-0.055
52,Shrub/Scrub,0.088,0.046,-0.042


## Impervious fraction

This is a different quantity from the developed class share and a better one.
The fractional impervious surface product carries a percent per pixel, which is
the direct input to NEH-630 Equations 9-1 and 9-2. The tabular route cannot do
this at all: it can only infer an impervious fraction from the assumed
midpoints of classes 21 to 24.

NoData is masked rather than unmasked here, because a NoData value of 250
averaged in as "250 percent impervious" would be silently catastrophic. The
mean is therefore over the covered part of the basin only.

What this number cannot tell you is how much of that impervious area is
hydraulically **connected** to the drainage system, which is the thing Equation
9-2 actually turns on. `cnkit` does not automate that split, and no Earth
Observation product settles it.

Expect two numbers in the region of fifteen to twenty five percent, with 2019
a little above 2001.

In [9]:
for year in (2001, 2019):
    print("%d mean impervious  %6.2f percent" % (year, basin.impervious(year)))

2001 mean impervious   16.44 percent
2019 mean impervious   18.24 percent


## The soils decision, which is the teaching moment

Every call in `cnkit` that needs a hydrologic soil group takes `soils=` as a
**required** keyword. There is no default and no fallback. Omitting it is a
`TypeError`.

That is a deliberate maintainer decision, not an oversight. The best CONUS
hydrologic soil group source is not in Earth Engine, so any default the package
picked would either make a hidden call outside Earth Engine or quietly use a
modelled global soil product. Both belong in a design calculation as a
decision made out loud.

Expect a `TypeError` whose message names both enabled values, their one-line
tradeoffs, and the disabled CSRL path.

In [10]:
try:
    basin.soil_groups()
except TypeError as exc:
    print(exc)

Basin.soil_groups is missing the required keyword `soils`.
soils= is a required keyword and has no default. Pass one of:
  soils='sda'          USDA Soil Data Access. True SSURGO dominant
                       condition hydrologic soil group at 30 m, authoritative
                       for CONUS. Makes one small call outside Earth Engine.
  soils='hihydrosoil'  HiHydroSoil v2.0 at 250 m. The only global option,
                       but a pedotransfer model output rather than a soil
                       survey. Experimental.
There is deliberately no default. The best CONUS source is not in Earth
Engine, so any default would either make a hidden call outside Earth
Engine or quietly substitute a modelled global soil product.
  soils='csrl' passed its live coverage and complete-reduction tests but
  is disabled for 1.1 because it is an 800 m community raster without
  embedded class-name metadata.


| `soils=` | Source | Native scale | Tradeoff |
| --- | --- | --- | --- |
| `'sda'` | gridded SSURGO map unit keys in Earth Engine, letters resolved by one USDA Soil Data Access query | 30 m | authoritative for CONUS, makes one small call outside Earth Engine |
| `'csrl'` | CSRL Soil Properties hydrologic group | 800 m | disabled pending a live Virginia coverage and reduction smoke test |
| `'hihydrosoil'` | HiHydroSoil v2.0 | 250 m | the only global option, but a pedotransfer model output rather than a soil survey |

The `'sda'` path is worth understanding because it is the one that is not
obvious. Shipping SSURGO polygons **into** Earth Engine would put thousands of
polygons through a 10 MB request ceiling. Instead the cross runs against the
gridded map unit key raster, which is already on the 30 m grid, and the only
thing that leaves Earth Engine is a short list of integers, resolved to
`muaggatt.hydgrpdcd` by one Soil Data Access query.

Now run the same basin with the authoritative 30 m CONUS source and the
experimental 250 m global source.

Expect both to be dominated by B and C on the Virginia Piedmont, and expect
them to disagree. Expect the `'sda'` table to carry a `(none mapped)` row of
roughly ten percent, because SSURGO maps urban land, made land, pits and open
water with no hydrologic soil group at all: that was about nineteen percent of
Fairfax County by area. Expect `'hihydrosoil'` to emit a
`CnkitExperimentalWarning` explaining that it is a modelled product.

In [11]:
sda_soils = basin.soil_groups(soils="sda")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    hihydro_soils = basin.soil_groups(soils="hihydrosoil")

print(caught[0].message)

/usr/local/lib/python3.12/dist-packages/cnkit/gee/basin.py:1111: UserWarning: Soil Data Access returned no hydrologic soil group for 218 of 375 map units in this basin, so that area is reported as unmapped. SSURGO maps urban land, made land, pits and open water without a group, which was roughly 19 percent of Fairfax County by area, so this is usually real rather than a fault.
  letters = self._soil_letters(spec, pct)


soils='hihydrosoil' is a pedotransfer model output derived from SoilGrids250m v2.0, not a soil survey. It is the only global hydrologic soil group raster in Earth Engine and it carries the correct NRCS dual group semantics, but a curve number built on it inherits the uncertainty of a modelled soil profile. The TR-55 tables it is being crossed with are a United States artefact; applying them outside the United States is a cnkit convention and not an agency product. It is being resampled from 250 m to the 30 m analysis grid, so the joint distribution is at best as good as the coarser layer.


In [12]:
comparison = (sda_soils.set_index("hsg")
              .join(hihydro_soils.set_index("hsg"), lsuffix="_sda", rsuffix="_hihydro",
                    how="outer")
              .fillna(0.0))
comparison["difference"] = comparison["pct_hihydro"] - comparison["pct_sda"]
comparison.round(2)

,pct_sda,pct_hihydro,difference
hsg,,,
(none mapped),21.27,0.00,-21.27
A,2.46,0.00,-2.46
B,17.72,0.00,-17.72
B/D,7.53,0.00,-7.53
C,48.72,17.61,-31.11
C/D,1.01,58.54,57.52
D,1.28,23.85,22.58


### The part of this you must not skip

The CSRL lookup is now verified against the California Soil Resource Lab's
published `hydgrp.csv`: 1=A, 2=A/D, 3=B, 4=B/D, 5=C, 6=C/D, 7=D and
8=unmapped. That corrected mapping is retained in the asset registry.

A correct lookup does not establish that the community Earth Engine mirror
covers Virginia or that its end-to-end reduction is sound. For that reason
`soils='csrl'` is disabled in 1.1.0 until the maintainer smoke test passes.

A wrong mapping does not raise. It does not warn. It returns a perfectly
well formed table of curve numbers **for the wrong soils**. Group A and group D
differ by roughly thirty curve numbers on the same land cover, so a permuted
mapping would move a composite by more than every other effect in this notebook
combined and would look entirely reasonable while doing it.

Do not bypass that guard in workshop or design work. The live test inspects
the asset directly without exposing the disabled public path.

In [13]:
print("source-verified CSRL lookup:", CSRL_HSG_CODES)
try:
    basin.soil_groups(soils="csrl")
except ValueError as exc:
    print("public guard:", exc)

source-verified CSRL lookup: {1: 'A', 2: 'A/D', 3: 'B', 4: 'B/D', 5: 'C', 6: 'C/D', 7: 'D', 8: '(none mapped)'}
public guard: soils='csrl' is disabled for the 1.1 release. Its source lookup, Virginia coverage and complete cnkit reduction passed live validation on 6 August 2026, but the owner kept it out of the public API because it is an 800 m community raster without embedded class-name metadata. Use soils='sda' for United States work or soils='hihydrosoil' for an explicitly experimental global path.


## The joint distribution, which is why this sub-package exists

The tabular route returns a land cover distribution and a soil distribution
**separately**. Crossing them requires assuming the two are independent, and
they are not: floodplains are wet soils and are also where the wetlands are,
hilltops are well drained and are also where the subdivisions went.

Earth Engine can cross the two rasters pixel by pixel and simply observe the
joint distribution. `joint_landcover_soils` packs the two single band images
into one integer band, `landcover * multiplier + soil`, computed in int64, and
takes one frequency histogram over it. Sixteen classes by four groups would be
64 masked reductions done the obvious way, against one here.

Expect: columns `nlcd`, `hsg`, `pct`, percentages summing to 100 over the whole
basin including unmapped cover and unmapped soil, so the table is a partition of
the basin rather than a partition of the part that happened to be classifiable.
Expect noticeably fewer rows than the full product of the two marginals, because
most cover and soil pairs do not actually occur.

In [14]:
joint = basin.joint_landcover_soils(2019, soils="sda")
print("rows %d, total %.4f percent" % (len(joint), joint["pct"].sum()))
joint.sort_values("pct", ascending=False).head(12)

rows 80, total 100.0000 percent


/usr/local/lib/python3.12/dist-packages/cnkit/gee/basin.py:1228: UserWarning: Soil Data Access returned no hydrologic soil group for 214 of 371 map units in this basin, so that area is reported as unmapped. SSURGO maps urban land, made land, pits and open water without a group, which was roughly 19 percent of Fairfax County by area, so this is usually real rather than a fault.
  letters = self._soil_letters(spec, soil_codes)


,nlcd,hsg,pct
10,21,C,26.999376
17,22,C,14.724036
35,41,B,8.783790
20,23,(none mapped),7.375942
8,21,B,5.558459
13,22,(none mapped),4.929193
6,21,(none mapped),4.223629
37,41,C,3.464467
36,41,B/D,2.918939
74,90,B/D,2.816172


## What the independence assumption is worth

Now the measurement. Take the **marginals out of the joint table itself**, so
nothing differs between the two calculations except the assumption: same
pixels, same year, same soils source, same boundary. Cross the marginals as if
land cover and soil group were independent, and compare the composite curve
number to the one from the observed joint.

Both go through `cnkit.lookup.composite_from_areas`. There is exactly one curve
number implementation in this library and neither this notebook nor
`cnkit.gee` contains any of it.

Expect the independence cross to produce many more rows, most of them tiny and
some of them pairs that do not exist on the ground. Expect the two composite
curve numbers to differ by roughly **4.5 units**, which is the number the
workshop measured. Hold on to it: the entire measured 2001 to 2019 land cover
change signal on this basin is 0.34 units, so the assumption is worth about
thirteen times the change, and notebook 3 shows an assumption worth twenty
times more still.

In [15]:
lc_marginal = joint.groupby("nlcd", as_index=False)["pct"].sum()
hsg_marginal = joint.groupby("hsg", as_index=False)["pct"].sum()

independent = pd.DataFrame(
    [{"nlcd": int(a.nlcd), "hsg": s.hsg, "pct": a.pct * s.pct / 100.0}
     for a in lc_marginal.itertuples() for s in hsg_marginal.itertuples()]
)
print("observed joint      %4d rows" % len(joint))
print("independence cross  %4d rows" % len(independent))

observed joint        80 rows
independence cross    91 rows


In [16]:
KW = dict(condition="fair", nlcd_col="nlcd", hsg_col="hsg", area_col="pct")
observed = composite_from_areas(joint, **KW)
assumed = composite_from_areas(independent, **KW)

print("=" * 64)
print("Difficult Run, 2019, fair condition, SSURGO soils")
print("-" * 64)
print("crossed assuming independence   CN = %8.4f" % assumed["cn_weighted_CN"])
print("observed joint distribution     CN = %8.4f" % observed["cn_weighted_CN"])
print("the independence assumption          %+8.4f curve numbers"
      % (assumed["cn_weighted_CN"] - observed["cn_weighted_CN"]))
print("-" * 64)
print("measured 2001 to 2019 change          %+8.4f curve numbers" % 0.34)
print("=" * 64)

Difficult Run, 2019, fair condition, SSURGO soils
----------------------------------------------------------------
crossed assuming independence   CN =  77.6638
observed joint distribution     CN =  75.7790
the independence assumption           +1.8848 curve numbers
----------------------------------------------------------------
measured 2001 to 2019 change           +0.3400 curve numbers


## The composite, and the soils source it depends on

`composite_cn` builds the area table and hands it to
`composite_from_areas`. It returns both weighting conventions, because they are
different averages and they disagree: `cn_weighted_CN` is the TR-55 Worksheet 2
area weighted mean of curve numbers, `cn_weighted_S` averages the retention S
instead. It also returns the provenance a reviewer will ask for, and
`percent_area_unmapped`, which is never dropped.

Read `condition` before you read the answer. TR-55 rows are soil cover
complexes carrying a hydrologic condition modifier that depends on ground cover
density, litter depth, grazing and compaction, and no Earth Observation product
measures any of those. That is notebook 3.

Expect a 2019 fair condition composite in the **middle seventies**; the
workshop's tabular route gives 75.4962 for the same basin and year. Expect the
Earth Engine answer not to reproduce that exactly and expect that to be
interesting rather than wrong: the gap is the independence assumption plus two
different NLCD products plus two different boundaries. Expect
`percent_area_unmapped` around ten percent, almost all of it SSURGO map units
with no group. The HiHydroSoil comparison is an uncertainty demonstration,
not a replacement for SSURGO on a CONUS design basin.

In [17]:
with warnings.catch_warnings(record=True):
    warnings.simplefilter("always")
    by_soils = {source: basin.composite_cn(2019, condition="fair", soils=source)
                for source in ("sda", "hihydrosoil")}

summary = pd.DataFrame(by_soils).T[
    ["cn_weighted_CN", "cn_weighted_S", "percent_area_unmapped",
     "percent_area_no_soil", "n_pairs", "scale_m"]
]
summary.round(4)

,cn_weighted_CN,cn_weighted_S,percent_area_unmapped,percent_area_no_soil,n_pairs,scale_m
sda,75.779032,73.87636,21.265543,21.265543,80,30.0
hihydrosoil,83.956409,83.648959,0.0,0.0,38,30.0


In [18]:
gap = (by_soils["hihydrosoil"]["cn_weighted_CN"] - by_soils["sda"]["cn_weighted_CN"])
print("250 m HiHydroSoil minus 30 m SSURGO   %+.4f curve numbers" % gap)
print()
print("assets used for soils='sda' :", by_soils["sda"]["assets"])
print("assets used for soils='hihydrosoil':", by_soils["hihydrosoil"]["assets"])

250 m HiHydroSoil minus 30 m SSURGO   +8.1774 curve numbers

assets used for soils='sda' : ('projects/sat-io/open-datasets/USGS/ANNUAL_NLCD/LANDCOVER', 'projects/sat-io/open-datasets/gNATSGO/raster/mukey')
assets used for soils='hihydrosoil': ('projects/sat-io/open-datasets/USGS/ANNUAL_NLCD/LANDCOVER', 'projects/sat-io/open-datasets/HiHydroSoilv2_0/Hydrologic_Soil_Group_250m')


## What to take from this

- `soils=` has no default anywhere in `cnkit`, and the `TypeError` you get for
  omitting it is the API refusing to make a resolution decision on your behalf.
- On CONUS use `soils='sda'`. HiHydroSoil is a modelled global product and
  should be labelled experimental in any result.
- The source-verified CSRL lookup is retained, but `soils='csrl'` is disabled
  until its Virginia footprint and full reduction pass the live smoke test.
- The joint distribution is the reason the Earth Engine layer exists. Crossing
  marginals assumes an independence that does not hold, and on this basin that
  assumption is worth about 4.5 curve numbers against a measured land cover
  change signal of 0.34.
- Unmapped area is reported, never dropped. A composite computed over ninety
  percent of a basin and printed as if it covered all of it is the kind of tidy
  number that gets believed.

**Next:** `03_changing_curve_number.ipynb` runs this across a span of years and
puts the one assumption that is larger than all of the above on the same axes.